# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 01.02 · Piloto opcional de longitud de chunks

Compara localmente ventanas de 15, 20, 25, 30 y 35 segundos con dos baselines CPU y permite elegir la longitud manualmente o aceptar una recomendación no productiva.

La selección de hiperparámetros usa exclusivamente `validation`; consultar `test` para elegir introduciría sesgo de selección [1]. Se promedia *average precision* de los cuatro daños por ser una medida informativa ante desbalance [2]. ComplementNB y SGD con TF-IDF reutilizan el mismo entrenador de los cuadernos posteriores y la implementación de scikit-learn [3]. La transferencia de etiquetas por mayor solapamiento temporal, la muestra enriquecida, la tolerancia absoluta de 0.02 AP y el proxy de costo `filas_train × modelos` son decisiones metodológicas locales. El resultado es orientativo, no una estimación productiva; `test` se muestra solo después y nunca participa en la recomendación.

**Contrato v2.1:** `SEGURO` + cuatro daños entrenados, incluida `ATAQUE_POR_GENERO_IDENTIDAD`. `SEGURO` es excluyente; los daños son multietiqueta. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')


## Controles opcionales y elección manual

In [ ]:
RUN_CHUNK_LENGTH_SMOKE_TEST=False
RUN_CHUNK_LENGTH_CONFIRMATORY_TEST=False
CANDIDATE_SECONDS=(15,20,25,30,35)
TOY_MODELS=('complement_nb','sgd_incremental')
TOY_VIDEO_LIMITS={'train':40,'validation':16,'test':16}
TOY_MAX_FEATURES=12000
CONFIRMATORY_MODELS=('complement_nb','logistic_regression','sgd_incremental')
CONFIRMATORY_VIDEO_LIMITS={'train':200,'validation':80,'test':80}
CONFIRMATORY_SEEDS=(20260805,20260817,20260829)
CONFIRMATORY_MAX_FEATURES=20000
MAX_VALIDATION_AP_DROP=0.02
MANUAL_CHUNK_SECONDS=30.0  # Puede elegirse cualquier valor positivo
USE_SMOKE_RECOMMENDATION=False
USE_CONFIRMATORY_RECOMMENDATION=False
APPLY_CHUNK_SELECTION=False  # Si es False, no mueve ningún dataset
from moderacion_peru.colab import prepare_local_bundle_input
from moderacion_peru.chunk_optimization import activate_chunking_configuration, run_chunk_length_confirmatory_test, run_chunk_length_smoke_test
from moderacion_peru.incremental import DEFAULT_CHUNKING_CONFIGURATION
import json
TRANSCRIPTS=ROOT/'datos/raw/transcripts_raw.jsonl'
CHUNKS=ROOT/'datos/processed/chunks_v2.jsonl'
DATASET_CHECKPOINT=prepare_local_bundle_input('dataset_5_salidas',project_root=ROOT)
DATASET=Path(DATASET_CHECKPOINT['path'])
PILOT_ROOT=ROOT/'resultados/pilotos/chunk_length'
RECOMMENDATION=PILOT_ROOT/'recommendation.json'
CONFIRMATORY_RECOMMENDATION=PILOT_ROOT/'confirmatory_recommendation.json'
show_summary('Configuración de pruebas', {'humo_rápido':RUN_CHUNK_LENGTH_SMOKE_TEST,'confirmatoria_corta':RUN_CHUNK_LENGTH_CONFIRMATORY_TEST,'longitudes':CANDIDATE_SECONDS,'dataset':DATASET,'aplicar_selección':APPLY_CHUNK_SELECTION}, tone='neutral')

## Prueba de humo local de extremo a extremo

In [ ]:
if RUN_CHUNK_LENGTH_SMOKE_TEST:
    smoke_result=run_chunk_length_smoke_test(TRANSCRIPTS,CHUNKS,DATASET,PILOT_ROOT,candidate_seconds=CANDIDATE_SECONDS,model_names=TOY_MODELS,video_limits=TOY_VIDEO_LIMITS,max_features=TOY_MAX_FEATURES,max_validation_ap_drop=MAX_VALIDATION_AP_DROP)
    show_result('Recomendación del piloto',smoke_result['recommendation'],tone='success')
    show_table('Comparación por longitud',smoke_result['comparisons'],limit=len(CANDIDATE_SECONDS))
else:
    show_callout('Piloto desactivado','Cambie RUN_CHUNK_LENGTH_SMOKE_TEST=True para entrenar diez baselines CPU pequeños. Los resultados se reanudan por firma.',tone='neutral')

## Confirmación corta pareada

In [ ]:
if RUN_CHUNK_LENGTH_CONFIRMATORY_TEST:
    confirmatory_result=run_chunk_length_confirmatory_test(TRANSCRIPTS,CHUNKS,DATASET,PILOT_ROOT,candidate_seconds=CANDIDATE_SECONDS,model_names=CONFIRMATORY_MODELS,video_limits=CONFIRMATORY_VIDEO_LIMITS,seeds=CONFIRMATORY_SEEDS,max_features=CONFIRMATORY_MAX_FEATURES)
    show_result('Recomendación confirmatoria',confirmatory_result['recommendation'],tone='success')
    show_table('Media y dispersión entre cohortes pareadas',confirmatory_result['aggregated_comparisons'],limit=len(CANDIDATE_SECONDS))
else:
    show_callout('Confirmación desactivada','Active RUN_CHUNK_LENGTH_CONFIRMATORY_TEST=True solo después del piloto rápido. Reentrena e infiere 45 baselines CPU: 5 longitudes × 3 modelos × 3 cohortes.',tone='neutral')

## Previsualización o activación reversible

In [ ]:
if USE_CONFIRMATORY_RECOMMENDATION:
    if not CONFIRMATORY_RECOMMENDATION.is_file():
        raise FileNotFoundError('Ejecute primero la confirmación corta o seleccione MANUAL_CHUNK_SECONDS')
    selected_seconds=float(json.loads(CONFIRMATORY_RECOMMENDATION.read_text(encoding='utf-8-sig'))['recommended_seconds'])
    selection_source='01_02_confirmatory_recommendation'
elif USE_SMOKE_RECOMMENDATION:
    if not RECOMMENDATION.is_file():
        raise FileNotFoundError('Ejecute primero el piloto o seleccione MANUAL_CHUNK_SECONDS')
    selected_seconds=float(json.loads(RECOMMENDATION.read_text(encoding='utf-8-sig'))['recommended_seconds'])
    selection_source='01_02_smoke_recommendation'
else:
    selected_seconds=float(MANUAL_CHUNK_SECONDS)
    selection_source='01_02_manual'
if selected_seconds <= 0:
    raise ValueError('MANUAL_CHUNK_SECONDS debe ser positivo')
selected_config={**DEFAULT_CHUNKING_CONFIGURATION,'max_seconds':selected_seconds}
if APPLY_CHUNK_SELECTION:
    activation=activate_chunking_configuration(ROOT,selected_config,source=selection_source)
    show_result('Configuración activada sin borrar derivados',activation,tone='success')
else:
    show_summary('Selección previsualizada',{'segundos':selected_seconds,'origen':selection_source,'acción':'Active APPLY_CHUNK_SELECTION=True; 01_03 materializará o restaurará esta firma.'},tone='neutral')

## Referencias

[1] G. C. Cawley and N. L. C. Talbot, "On Over-Fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation," J. Mach. Learn. Res., vol. 11, pp. 2079–2107, 2010.

[2] T. Saito and M. Rehmsmeier, "The Precision-Recall Plot Is More Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets," PLOS ONE, vol. 10, no. 3, Art. no. e0118432, 2015, doi: 10.1371/journal.pone.0118432.

[3] F. Pedregosa, G. Varoquaux, A. Gramfort, et al., "Scikit-Learn: Machine Learning in Python," J. Mach. Learn. Res., vol. 12, pp. 2825–2830, 2011. [Online]. Available: https://www.jmlr.org/papers/v12/pedregosa11a.html